In [ ]:
!pip install wilds transformers datasets scikit-learn accelerate tqdm
!pip install -q datasets

In [ ]:
%%writefile dedier_multinli.py
import argparse, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm

# ────────────────────────── helpers ─────────────────────────────────
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def pick_device(pref="cuda:0"):
    if pref.startswith("cuda") and not torch.cuda.is_available():
        print("CUDA unavailable — using CPU")
        return torch.device("cpu")
    return torch.device(pref)

softmax = nn.Softmax(dim=1)

class Projector(nn.Module):
    def __init__(self, hidden_dim: int, num_classes: int = 3):
        super().__init__()
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, h): return self.fc(h)

@torch.no_grad()
def laplace_cov(projector: Projector, features: torch.Tensor, epsilon=1e-5):
    weight_matrix = projector.fc.weight
    centered_features = features - features.mean(dim=0, keepdim=True)

    feature_covariance = centered_features.T @ centered_features
    feature_covariance /= max(1, features.size(0) - 1)

    projected_covariance = weight_matrix @ feature_covariance @ weight_matrix.T

    projected_covariance = 0.5 * (projected_covariance + projected_covariance.T)

    min_eigenvalue = torch.linalg.eigvalsh(projected_covariance).min()
    if min_eigenvalue < epsilon:
        identity = torch.eye(projected_covariance.size(0), device=projected_covariance.device)
        projected_covariance += (epsilon - min_eigenvalue) * identity

    return projected_covariance

# ───────────────────── dataset construction ─────────────────────────
def has_neg(text: str) -> int:
    txt = text.lower()
    return int(any(t in txt for t in [" not ", "n't", " never ", " no "]))

def build_rows(split):
    rows=[]
    for ex in split:
        if ex["label"] == -1: continue
        rows.append({
            "text1": ex["premise"],
            "text2": ex["hypothesis"],
            "y":     ex["label"],
            "meta":  torch.tensor([ex["label"], has_neg(ex["hypothesis"])])
        })
    return rows

class NLIDataset(Dataset):
    def __init__(self, rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self,i): return self.rows[i]

def build_collate(tok):
    def coll(batch):
        t1,t2,y,m=[],[],[],[]
        for d in batch:
            t1.append(d["text1"]); t2.append(d["text2"])
            y.append(d["y"]);      m.append(d["meta"])
        enc = tok(t1,t2,padding=True,truncation=True,return_tensors="pt")
        return enc["input_ids"], enc["attention_mask"], \
               torch.tensor(y), torch.stack(m)
    return coll

# ──────────────────── training / evaluation ────────────────────────
def train_epoch(model,loader,opt,sched,device,*,log_int=100,tag=""):
    model.train(); run=0; seen=0
    for i,(inp,mask,lbl,_) in enumerate(tqdm(loader,leave=False)):
        inp,mask,lbl = inp.to(device),mask.to(device),lbl.to(device)
        loss = model(input_ids=inp,attention_mask=mask,labels=lbl).loss
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        run += loss.item()*lbl.size(0); seen += lbl.size(0)
        if (i+1)%log_int==0:
            print(f"{tag} step {i+1}/{len(loader)}  loss {run/seen:.4f}")
    return run/seen

@torch.no_grad()
def evaluate(model,loader,device):
    model.eval(); P,Y,G=[],[],[]
    for inp,mask,lbl,meta in tqdm(loader,leave=False):
        inp,mask=inp.to(device),mask.to(device)
        logits = model(input_ids=inp,attention_mask=mask).logits
        P.append(logits.argmax(1).cpu()); Y.append(lbl); G.append(meta[:,1])
    P,Y,G = torch.cat(P),torch.cat(Y),torch.cat(G)
    acc = accuracy_score(Y,P)
    per_group={int(g): accuracy_score(Y[G==g],P[G==g]) for g in torch.unique(G)}
    return acc, per_group

# ─────────────────────────── main ───────────────────────────────────
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--output_dir", default="./outputs")
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--epochs_teacher", type=int, default=3)
    ap.add_argument("--epochs_student", type=int, default=5)
    ap.add_argument("--batch_size", type=int, default=16)
    ap.add_argument("--lr_teacher", type=float, default=2e-5)
    ap.add_argument("--lr_student", type=float, default=2e-5)
    ap.add_argument("--alpha", type=float, default=0.2)
    ap.add_argument("--beta",  type=float, default=3.0)
    ap.add_argument("--log_interval", type=int, default=100)
    # new flags
    ap.add_argument("--debug_one_batch", action="store_true")
    ap.add_argument("--teacher_path", type=str, default="")
    args = ap.parse_args()

    set_seed(args.seed)
    device = pick_device(args.device)
    Path(args.output_dir).mkdir(parents=True, exist_ok=True)

    # 0. dataset
    print("Downloading / loading MultiNLI …")
    hf = load_dataset("multi_nli")
    splits={"train":hf["train"],
            "val":hf["validation_matched"],
            "test":hf["validation_mismatched"]}
    train_set,val_set,test_set = map(lambda s:NLIDataset(build_rows(s)),
                                     (splits["train"],splits["val"],splits["test"]))
    print(f"✓ Loaded: {len(train_set)} train / {len(val_set)} val / {len(test_set)} test")
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    coll = build_collate(tok)
    train_loader = DataLoader(train_set,batch_size=args.batch_size,
                              shuffle=True,collate_fn=coll)
    val_loader   = DataLoader(val_set,batch_size=args.batch_size,
                              collate_fn=coll)
    test_loader  = DataLoader(test_set,batch_size=args.batch_size,
                              collate_fn=coll)

    if args.debug_one_batch:
        train_loader=[next(iter(train_loader))]
        val_loader  =[next(iter(val_loader))]
        test_loader =[next(iter(test_loader))]
        print("DEBUG-ONE-BATCH MODE: loaders truncated to 1 batch")

    # 1. Teacher
    teacher = AutoModelForSequenceClassification.from_pretrained(
                "bert-base-uncased",num_labels=3).to(device)
    if args.teacher_path and Path(args.teacher_path).exists():
        teacher.load_state_dict(torch.load(args.teacher_path, map_location=device))
        print(f"Teacher loaded from {args.teacher_path}")
    else:
        t_opt  = optim.AdamW(teacher.parameters(), lr=args.lr_teacher)
        t_sched= get_linear_schedule_with_warmup(
                    t_opt,0,len(train_loader)*args.epochs_teacher)
        for ep in range(args.epochs_teacher):
            tloss=train_epoch(teacher,train_loader,t_opt,t_sched,device,
                              log_int=args.log_interval,tag="Teacher")
            vacc,_=evaluate(teacher,val_loader,device)
            print(f"Teacher {ep+1}/{args.epochs_teacher}: loss {tloss:.4f} val-acc {vacc:.3f}")
        save_path=Path(args.output_dir)/"teacher.pt"
        torch.save(teacher.state_dict(),save_path)
        print(f"Teacher saved to {save_path}")

    # 2. Projector
    projector=Projector(teacher.config.hidden_size,3).to(device)
    proj_opt =optim.AdamW(projector.parameters(),lr=1e-4)

    # 3. Student
    student = AutoModelForSequenceClassification.from_pretrained(
                "distilbert-base-uncased",num_labels=3).to(device)
    s_opt  = optim.AdamW(student.parameters(), lr=args.lr_student)
    s_sched= get_linear_schedule_with_warmup(
                s_opt,0,len(train_loader)*args.epochs_student)

    margins, group_ids = [], []

    for ep in range(args.epochs_student):
        student.train(); teacher.eval(); projector.train()
        kd_tot=ce_tot=seen=0
        for b,(inp,mask,lbl,meta) in enumerate(tqdm(train_loader,leave=False)):
            inp,mask,lbl = inp.to(device),mask.to(device),lbl.to(device)

            # (a) CLS feats from student layer 3
            with torch.no_grad():
                out = student.distilbert(input_ids=inp, attention_mask=mask,
                                        return_dict=True, output_hidden_states=True)
                feats = out.hidden_states[3][:, 0]  # Layer 3 CLS token

            # (b) projector update (one backward)
            proj_logits = projector(feats)
            proj_loss = nn.CrossEntropyLoss()(proj_logits, lbl)
            proj_opt.zero_grad(); proj_loss.backward(); proj_opt.step()

            # (c) entropy margin (detach proj_logits)
            with torch.no_grad():
                cov_matrix = laplace_cov(projector, feats)
                mvn = torch.distributions.MultivariateNormal(
                        proj_logits.detach(), covariance_matrix=cov_matrix)
                probs = softmax(mvn.rsample((20,))).mean(0)
                ent = -(probs * probs.log()).sum(1).cpu()  # (B,)
            margins.extend(ent.tolist())
            group_ids.extend(meta[:, 1].tolist())

            # (d) KD+CE with weights
            with torch.no_grad():
                t_logits = teacher(input_ids=inp,attention_mask=mask).logits
            s_logits = student(input_ids=inp,attention_mask=mask).logits
            kd = nn.KLDivLoss(reduction="batchmean")(
                    nn.LogSoftmax(dim=1)(s_logits/2.), softmax(t_logits/2.))*4.
            ce = nn.CrossEntropyLoss()(s_logits, lbl)
            w  = torch.exp(args.beta*(ent.to(device)**args.alpha))
            loss = (kd*w).mean() + ce
            s_opt.zero_grad(); loss.backward(); s_opt.step(); s_sched.step()

            kd_tot += kd.mean().item()*lbl.size(0)
            ce_tot += ce.item()*lbl.size(0); seen += lbl.size(0)
            if (b+1)%args.log_interval==0:
                print(f"  batch {b+1}  proj {proj_loss.item():.4f} "
                      f"kd {kd.mean():.4f}  ce {ce.item():.4f} "
                      f"margin {ent.mean():.3f}")

        print(f"Student epoch {ep+1}: KD {kd_tot/seen:.4f} "
              f"CE {ce_tot/seen:.4f}  avg_margin {np.mean(margins):.3f}")

    torch.save(student.state_dict(), Path(args.output_dir)/"student.pt")

    # 4. Evaluation
    print("\nFinal evaluation:")
    t_acc,t_g = evaluate(teacher,test_loader,device)
    s_acc,s_g = evaluate(student,test_loader,device)
    print(f"Teacher avg acc: {t_acc:.3f}   Student avg acc: {s_acc:.3f}")
    worst = min(t_g, key=t_g.get)
    print(f"Worst group {worst}: teacher {t_g[worst]:.3f} "
          f"student {s_g[worst]:.3f}")

    # 5. Margin stats
    g_marg = defaultdict(list)
    for m,gid in zip(margins,group_ids): g_marg[int(gid)].append(m)
    print("\nAvg predictive entropy (teacher Laplace) per group:")
    for g in sorted(g_marg):
        print(f"  group {g}: {np.mean(g_marg[g]):.3f}")
    print(f"Worst-group margin ({worst}): {np.mean(g_marg[worst]):.3f}")

if __name__=="__main__":
    main()

In [ ]:
!python dedier_multinli.py --output_dir ./outputs --device cuda:0 --epochs_teacher 3 --epochs_student 5 --batch_size 16 --log_interval 50

In [ ]:

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
import random
from collections import defaultdict

# ────────────────────────── setup ──────────────────────────────────────
DEVICE  = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
SAMPLES = 512   # number of validation examples to sample

# ───────────────────────── helpers ────────────────────────────────────
softmax = nn.Softmax(dim=1)

def has_neg(text: str) -> int:
    t = text.lower()
    return int(any(tok in t for tok in [" not ", "n't", " never ", " no "]))

def confidence_margin(logits: torch.Tensor) -> np.ndarray:
    """Return p_max - p_2nd for each example."""
    p = softmax(logits)
    top2 = torch.topk(p, 2, dim=1).values   # (B,2)
    return (top2[:,0] - top2[:,1]).detach().cpu().numpy()

def make_4d_mask(attn_mask_2d: torch.Tensor):
    m = attn_mask_2d.to(torch.bool)  # [B,S]
    return m[:,None,None,:].expand(-1,1,m.size(1),m.size(1))

def extract_cls_per_layer_bert(model, input_ids, attn2d):
    mask4 = make_4d_mask(attn2d)
    h = model.bert.embeddings(input_ids=input_ids)
    cls_list = []
    for layer in model.bert.encoder.layer:
        h = layer(h, attention_mask=mask4)[0]
        cls_list.append(h[:,0])
    return cls_list

def extract_cls_per_layer_distil(model, input_ids, attn2d):
    mask4 = make_4d_mask(attn2d)
    h = model.distilbert.embeddings(input_ids=input_ids)
    cls_list = []
    for layer in model.distilbert.transformer.layer:
        h = layer(h, attn_mask=mask4)[0]
        cls_list.append(h[:,0])
    return cls_list

# ─────────────────────────── main ────────────────────────────────────
def main():
    torch.manual_seed(0)
    np.random.seed(0)
    random.seed(0)

    # 1. Load data
    ds = load_dataset("multi_nli")["validation_matched"].select(range(SAMPLES))
    premises, hypos = ds["premise"], ds["hypothesis"]
    labels = ds["label"]
    negs   = [has_neg(h) for h in hypos]
    genres = ds["genre"]

    # 2. Tokenize
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    enc = tok(premises, hypos, padding=True, truncation=True, return_tensors="pt")
    input_ids = enc["input_ids"].to(DEVICE)
    attn2d    = enc["attention_mask"].to(DEVICE)
    labels_t  = torch.tensor(labels).to(DEVICE)
    negs_t    = torch.tensor(negs).to(DEVICE)

    # 3. Load models
    teacher = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased", num_labels=3
    ).to(DEVICE)
    teacher.load_state_dict(torch.load("./outputs/teacher.pt", map_location=DEVICE))
    teacher.eval()

    student = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=3
    ).to(DEVICE)
    student.load_state_dict(torch.load("./outputs/student.pt", map_location=DEVICE))
    student.eval()

    # 4. Extract CLS features per layer
    with torch.no_grad():
        cls_t = extract_cls_per_layer_bert(teacher, input_ids, attn2d)
        cls_s = extract_cls_per_layer_distil(student, input_ids, attn2d)

    # 5. Compute confidence margins per layer
    def compute_margins(cls_list):
        margins = []
        for feat in cls_list:
            head   = nn.Linear(feat.size(1), 3, bias=False).to(DEVICE)
            logits = head(feat)
            margins.append(confidence_margin(logits))
        return margins

    marg_t = compute_margins(cls_t)
    marg_s = compute_margins(cls_s)

    # 6. Define worst vs others
    worst_mask = (labels_t.cpu().numpy() == 2) & (negs_t.cpu().numpy() == 1)
    other_mask = ~worst_mask

    avg_t_worst = [m[worst_mask].mean() for m in marg_t]
    avg_t_other = [m[other_mask].mean() for m in marg_t]
    avg_s_worst = [m[worst_mask].mean() for m in marg_s]
    avg_s_other = [m[other_mask].mean() for m in marg_s]

    # 7. Plot confidence margin vs layer
    layers_t = np.arange(1, len(avg_t_worst)+1)
    layers_s = np.linspace(1, len(avg_t_worst), len(avg_s_worst))

    plt.figure(figsize=(6,4))
    plt.plot(layers_t, avg_t_worst,  "o-", label="Teacher Worst",  color="C0")
    plt.plot(layers_t, avg_t_other,  "s--",label="Teacher Others", color="C1")
    plt.plot(layers_s, avg_s_worst,  "o-", label="Student Worst",  color="C2")
    plt.plot(layers_s, avg_s_other,  "s--",label="Student Others", color="C3")
    plt.xlabel("Layer")
    plt.ylabel("Confidence Margin\n$(p_{\\max}-p_{2nd})$")
    plt.title("Val: Worst vs Others by Layer")
    plt.legend(); plt.tight_layout(); plt.show()

    # 8. Compute avg confidence on wrong predictions
    with torch.no_grad():
        # teacher
        tl = teacher(input_ids=input_ids, attention_mask=attn2d.bool()).logits
        tp = softmax(tl); tp_pred = tp.argmax(1)
        wrong_t = (tp_pred != labels_t)

        # student
        sl = student(input_ids=input_ids, attention_mask=attn2d.bool()).logits
        sp = softmax(sl); sp_pred = sp.argmax(1)
        wrong_s = (sp_pred != labels_t)

        def avg_conf(probs, wrong_mask, group_mask=None):
            mask = wrong_mask
            if group_mask is not None:
                mask &= torch.tensor(group_mask, device=probs.device)
            preds = probs[mask].argmax(1)
            confs = probs[mask].gather(1, preds.unsqueeze(1)).squeeze(1)
            return confs.mean().item()

        avg_t_all   = avg_conf(tp, wrong_t)
        avg_t_wg    = avg_conf(tp, wrong_t, worst_mask)
        avg_s_all   = avg_conf(sp, wrong_s)
        avg_s_wg    = avg_conf(sp, wrong_s, worst_mask)

    print(f"\nTeacher avg confidence on wrong preds (all):   {avg_t_all:.3f}")
    print(f"Teacher avg confidence on wrong preds (worst): {avg_t_wg:.3f}")
    print(f"Student avg confidence on wrong preds (all):   {avg_s_all:.3f}")
    print(f"Student avg confidence on wrong preds (worst): {avg_s_wg:.3f}")

    # 9. Bar plot for wrong‐pred confidence
    labels = ["Teacher","Student"]
    all_conf   = [avg_t_all, avg_s_all]
    worst_conf = [avg_t_wg,  avg_s_wg]

    x, w = np.arange(2), 0.35
    fig, ax = plt.subplots(figsize=(5,4))
    ax.bar(x-w/2, all_conf,   w, label="All wrong")
    ax.bar(x+w/2, worst_conf, w, label="Worst-group wrong")
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("Avg Confidence on Wrong")
    ax.set_title("Wrong‐Prediction Confidence")
    ax.legend(); plt.tight_layout(); plt.show()

    wrong_margin_t = [m[wrong_t.cpu().numpy()].mean() if wrong_t.any() else np.nan for m in marg_t]
    wrong_margin_s = [m[wrong_s.cpu().numpy()].mean() if wrong_s.any() else np.nan for m in marg_s]

    plt.figure(figsize=(6,4))
    plt.plot(layers_t, wrong_margin_t, "o-", label="Teacher wrong",  color="C4")
    plt.plot(layers_s, wrong_margin_s, "s--",label="Student wrong", color="C5")
    plt.xlabel("Layer")
    plt.ylabel("Avg Conf Margin on Wrong")
    plt.title("Wrong‐Prediction Confidence Margin per Layer")
    plt.legend(); plt.tight_layout(); plt.show()

    genre_marg = defaultdict(list)
    for g, ent in zip(genres, marg_t[-1]):
        genre_marg[g].append(ent)
    print("\nTeacher last-layer confidence margin by genre:")
    for g in sorted(genre_marg):
        print(f"  {g:12s}: {np.mean(genre_marg[g]):.3f}")

if __name__=="__main__":
    main()